# 14 · Geographic building scenes with MapLibre

**Question:** How can a 3D map preserve geographic position and honest building heights?

**Runtime:** Python / Pyodide in JupyterLite. Choose **Run → Run All Cells**.
Each notebook is independent; start with a fresh kernel. Initial runtime/package downloads need internet.
All city geometry, residents, sensor observations and parameters are **synthetic educational fixtures**.
The longitude/latitude anchor is near Gaborone, but these are not mapped Gaborone assets or a validated city twin.

**Astra experiment:** Ask Astra to audit a 3D scene against its GeoJSON, then have Codex add a visual control with explicit data invariants.
Run the numerical baseline first, then give Astra the exported evidence and the prompt below.
The baseline does not call or run GPT-6 Astra. No API key is needed.


In [ ]:
import sys, json, math, time
from pathlib import Path
if sys.platform == "emscripten":
    import piplite
    await piplite.install(["numpy", "matplotlib"])
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from twin import *
style()
print("Runtime:", sys.platform, "| data: synthetic | seed: 42")


## From footprint to extrusion · 30 minutes
**Learn:** create WGS84 GeoJSON, distinguish local coordinates from geographic coordinates, and use a
MapLibre `fill-extrusion` layer. Horizontal coordinates are longitude/latitude; heights are metres above
an assumed flat ground plane. Floor height is invented, not measured. Change `floor_height_m` and rerun.
MapLibre projects the GeoJSON for display; do not put local east/north metres directly into its coordinates.
**Predict:** should doubling floor height change population or footprint area?


In [ ]:
from scenes import building_features, export_scene, scene_frame
c=city(); pop=allocate(c['capacity'],1200); floor_height_m=3.
buildings=building_features(c,pop,floor_height_m)
nodes,graph=network()
roads=dict(type='FeatureCollection',features=[dict(type='Feature',properties={},geometry=dict(type='LineString',
    coordinates=[[float(v) for v in lonlat(*nodes[u])],[float(v) for v in lonlat(*nodes[v])]]))
    for u in graph for v,w in graph[u] if u<v])
assert sum(f['properties']['population'] for f in buildings['features'])==1200
assert all(f['geometry']['coordinates'][0][0]==f['geometry']['coordinates'][0][-1] for f in buildings['features'])
heights=np.array([f['properties']['height_m'] for f in buildings['features']])
assert np.allclose(heights,c['floors']*floor_height_m)
center=[float(v) for v in lonlat(1000,1000)]
payload=dict(buildings=buildings,roads=roads,center=center,synthetic=True,vertical_units='metres above flat ground')
export_json('14_buildings.geojson',buildings)
print('Buildings:',len(heights),'| physical heights:',sorted(set(heights)),'m | residents:',pop.sum())


In [ ]:
fig,axes=plt.subplots(1,2,figsize=(12,5))
dots=axes[0].scatter(*c['xy'].T,c=heights,s=25,cmap='viridis'); map_axes(axes[0],'Plan-view height audit')
fig.colorbar(dots,ax=axes[0],label='Assumed height (m)')
axes[1].hist(heights,bins=np.arange(.5,4.5)*floor_height_m,color='#087f8c')
axes[1].set(xlabel='Assumed building height (m)',ylabel='Buildings',title='Unexaggerated height distribution')
plt.tight_layout(); plt.show()


## Use and share the interactive scene
The scene appears below in a trusted notebook output. If your notebook viewer blocks the iframe,
download the HTML from `exports/` and open it in a browser. All data are embedded, but the pinned
JavaScript/CSS renderer still downloads from a CDN; this is not fully offline. WebGL must be available.
The static plot and numerical outputs remain usable when the renderer cannot load.

Camera rotation and vertical exaggeration change the view, not the underlying data. Keep a plan view
beside a perspective view to detect occlusion. Download the notebook, scene and evidence JSON before
clearing browser storage. Do not label synthetic geometry as observed Gaborone buildings.


In [ ]:
document=export_scene('14_maplibre_scene.html','maplibre',payload)
display(scene_frame(document,'MapLibre synthetic buildings in 3D'))


## Astra interpretation and a Codex implementation task
**Astra:** send `14_buildings.geojson` plus a screenshot at 1× and 5× exaggeration.
Ask: “Which differences come from the camera or exaggeration? Which heights are supported by the data?
List the coordinate order, height units and assumptions. Do not infer occupancy or neighborhood deprivation
from appearance.” Score numerical claims against the GeoJSON, and mark screenshot-only impressions separately.

**Codex:** “Add a minimum-floor filter to `content/scene_templates/maplibre.html`. Show the visible count,
preserve IDs and raw heights, and keep the accessible table consistent with the filter. Test zero matches
and restoring all buildings. Do not rescale heights to make the scene look more dramatic.”

**Acceptance:** 144 original IDs; population total 1,200 before filtering; extrusion equals floors × floor height ×
display exaggeration; changing camera leaves data unchanged. Copy the exact prompt, response, model/date and
screenshots into your learning record. The initial notebook contains no measured Astra response.

Reference: [MapLibre 3D buildings](https://maplibre.org/maplibre-gl-js/docs/examples/display-buildings-in-3d/).
Our scene uses original GeoJSON and an empty style rather than the example's external tile source.


## Extend with Codex or Astra
Import a small licensed footprint extract with documented height units. Report missing heights explicitly and keep the source IDs, dates and attribution. Add terrain only after defining whether heights are above ground or a vertical datum.

Keep the baseline seed and tests fixed while changing one assumption. Record the model name,
date, exact prompt, response and error metrics. Generated code must pass the same checks.
Download files in `exports/` and your edited notebook; browser storage does not commit changes to GitHub.
See [the project guide](https://github.com/jltobias/JupyterLite-GPT-6-Astra-Geospatial/blob/main/docs/GUIDE.md)
for capabilities, evidence, sources and the route to real data.
